# Graph Neural Networks (message passing)

_Modern Deep Learning & AI_

**Each node updates itself by averaging its neighbours, then squishing. Do that a few times and the whole graph 'talks'.**

A graph is dots (nodes) joined by lines (edges). Think of friends and friendships, or atoms and bonds.

     A Graph Neural Network gives every node a small list of numbers (its features). Then it lets nodes share with their neighbours.

---

This notebook is a practice scaffold for the **Graph Neural Networks (message passing)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)   # shared weight matrix W

    def forward(self, H, A_hat):
        # H: (N, in_dim) node features ; A_hat: (N, N) row-normalized adjacency
        agg = A_hat @ H                          # mean of each node's neighbours
        return torch.relu(self.lin(agg))         # sigma(W . agg)

# tiny synthetic graph: 4 nodes, a couple of edges (+ self-loops)
A = torch.tensor([[1., 1., 0., 1.],
                  [1., 1., 1., 0.],
                  [0., 1., 1., 1.],
                  [1., 0., 1., 1.]])
A_hat = A / A.sum(dim=1, keepdim=True)           # normalize rows -> averaging
H = torch.randn(4, 3)                            # 4 nodes, 3 features each

layer = GCNLayer(in_dim=3, out_dim=2)
H2 = layer(H, A_hat)                             # (4, 2) updated features
print(H2.shape)

## Visualize the data & results

_On the real Zachary karate-club graph, what happens to each node's features after one round of message passing?_

In [ ]:
import numpy as np

# Real Zachary karate-club edge list (Zachary 1977), 34 nodes, 0-indexed.
edges = [(0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
(0,12),(0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),
(1,19),(1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),
(3,7),(3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),
(8,33),(9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),
(19,33),(20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),
(23,33),(24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),
(28,33),(29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33)]
n = 34
A = np.zeros((n, n))
for i, j in edges:
    A[i, j] = 1.0; A[j, i] = 1.0

# Two REAL structural node features: degree and triangle count.
deg = A.sum(1)
tri = np.array([(A @ A * A)[i].sum() / 2 for i in range(n)])
H = np.column_stack([deg, tri])

# One GCN layer = mean over self + neighbours (symmetric self-loop, row-normalized).
A_hat = A + np.eye(n)
H2 = (A_hat / A_hat.sum(1, keepdims=True)) @ H

sel = [0, 33, 1, 2, 32, 3]
names = ['Mr Hi (0)', 'Officer (33)', 'Node 1', 'Node 2', 'Node 32', 'Node 3']
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
for a, M, t in [(ax[0], H[sel], 'BEFORE'), (ax[1], H2[sel], 'AFTER one GCN layer')]:
    a.imshow(M, cmap='viridis', aspect='auto')
    a.set_xticks([0, 1]); a.set_xticklabels(['degree', 'triangles'])
    a.set_yticks(range(6)); a.set_yticklabels(names); a.set_title(t)
    for i in range(6):
        for j in range(2):
            a.text(j, i, format(M[i, j], '.2f'), ha='center', va='center', color='w')
plt.tight_layout(); plt.show()